# AdaMF-MAT reproduction (Level 2), runs top to bottom

Before **Save Version -> Save & Run All**: Settings -> Accelerator = GPU T4, **Internet = On**. No datasets need attaching. Output: `/kaggle/working/report.txt`. A failing stage is logged and the run continues.

In [ ]:
# ===== CONFIG + HELPERS (edit only these 3 numbers) =====
EPOCHS_SMOKE = 3        # quick check that code+data work
EPOCHS_FULL = 1000      # authors' scripts use 1000
TIMEOUT_H = 5           # per full run; Kaggle commit limit is ~12h, 2 datasets
EMB_PATH = "/kaggle/input/datasets/khnhnguyninhquang/adam-mat/AdaMF-MAT"  # folder with the downloaded embeddings, e.g. /kaggle/input/<your-dataset>/ ; "" = try Google Drive
import os, subprocess, shutil, glob, time, re, traceback
R, OUT = "/kaggle/working/AdaMF-MAT", "/kaggle/working"
REPORT = f"{OUT}/report.txt"
import torch
N_GPU = torch.cuda.device_count()
STATE = {"smoke_ok": False}
def log(*a):
    s = " ".join(str(x) for x in a)
    print(s, flush=True)
    open(REPORT, "a").write(s + "\n")
def sh(cmd, cwd=None, timeout=None):
    """run shell cmd, never raises; returns (returncode, last 4000 chars of output)"""
    try:
        p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, timeout=timeout)
        return p.returncode, (p.stdout + p.stderr)[-4000:]
    except subprocess.TimeoutExpired as e:
        return -9, "TIMEOUT " + str(e)[-500:]
def stage(name):
    def deco(fn):
        log(f"\n######## {name} ########")
        t = time.time()
        try:
            fn()
        except Exception:
            log("STAGE FAILED:", traceback.format_exc()[-2500:])
        log(f"-- {name} took {time.time()-t:.0f}s")
        return fn
    return deco
open(REPORT, "w").write("AdaMF-MAT reproduction report\n")

In [ ]:
@stage("1 clone + environment")
def _():
    if not os.path.exists(R):
        rc, o = sh(f"git clone -q https://github.com/zjukg/AdaMF-MAT {R}")
        log("clone rc", rc, o)
    log(sh(f"ls {R} && cat {R}/requirements.txt")[1])
    log(sh("python -V")[1])
    log(sh("python -c \"import torch, numpy, sklearn, tqdm; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'numpy', numpy.__version__)\"")[1])
    log(sh("nvidia-smi -L")[1])

In [ ]:
@stage("2 embeddings (uploaded folder if EMB_PATH set, else authors' Google Drive)")
def _():
    os.makedirs(f"{R}/embeddings", exist_ok=True)
    if EMB_PATH and os.path.isdir(EMB_PATH):
        log("copying embeddings from", EMB_PATH)
        log(sh(f"cp -r {EMB_PATH.rstrip('/')}/. {R}/embeddings/")[1])
    else:
        log("EMB_PATH not set or missing -> trying gdown")
        log(sh("pip install -q gdown")[1])
        url = "https://drive.google.com/drive/folders/1UJSfnb8DEx2s-k8zaQx1fWUw5f45GBpI"
        rc, o = sh(f'gdown --folder "{url}" -O {R}/embeddings', timeout=3600)
        log("gdown rc", rc, o[-1500:])
    # flatten: if everything landed in one nested folder, move it up
    sub = [d for d in glob.glob(f"{R}/embeddings/*") if os.path.isdir(d)]
    files = [f for f in glob.glob(f"{R}/embeddings/*") if os.path.isfile(f)]
    if sub and not files and len(sub) == 1:
        for f in glob.glob(sub[0] + "/*"):
            shutil.move(f, f"{R}/embeddings/")
    log(sh(f"ls -laR {R}/embeddings | head -60")[1])

In [ ]:
@stage("3 repo diagnostics (where the code expects data)")
def _():
    log(sh(f"ls -R {R}/benchmarks | head -60")[1])
    log(sh(f"cat {R}/args.py")[1])
    log(sh(f"grep -rn 'embeddings\\|benchmarks' --include=*.py {R} | head -30")[1])

In [ ]:
def run_adamf(data, dim, margin, epochs, tag, timeout_s=None, gpu=0, wait=True):
    os.makedirs(f"{R}/checkpoint", exist_ok=True)   # repo crashes at save if this folder is missing
    save = f"./checkpoint/{data}-{tag}"
    logf = f"{OUT}/{data}-{tag}.log"
    tmo = f"timeout {timeout_s} " if timeout_s else ""
    cmd = (f"CUDA_VISIBLE_DEVICES={gpu} {tmo}python run_adamf_mat.py -dataset={data} -batch_size=1024 -margin={margin} "
           f"-epoch={epochs} -dim={dim} -mu=0 -save={save} -neg_num=128 -learning_rate=2e-5 > {logf} 2>&1")
    t = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=R)
    if not wait:
        return p, logf, t
    rc = p.wait()
    log(f"{data} {tag}: rc={rc} time={time.time()-t:.0f}s\n--- log tail ---\n{open(logf).read()[-3500:]}")
    return rc

@stage("4 smoke test (MKG-W, few epochs; must reach the final test metrics)")
def _():
    rc = run_adamf("MKG-W", 200, 12, EPOCHS_SMOKE, "smoke", timeout_s=1800)
    STATE["smoke_ok"] = (rc == 0)
    log("SMOKE OK" if rc == 0 else "SMOKE FAILED -> full runs skipped; read the log tail above")

In [ ]:
@stage("5 full runs: MKG-W on GPU0 and MKG-Y on GPU1, in parallel")
def _():
    if not STATE["smoke_ok"]:
        log("skipped (smoke failed)")
        return
    sec = int(TIMEOUT_H * 3600)
    jobs = [("MKG-W", 12, 0), ("MKG-Y", 4, 1 if N_GPU > 1 else 0)]
    procs = [(d, *run_adamf(d, 200, m, EPOCHS_FULL, "full", timeout_s=sec, gpu=g, wait=False)) for d, m, g in jobs]
    for d, p, logf, t in procs:
        rc = p.wait()
        log(f"{d} full: rc={rc} time={time.time()-t:.0f}s\n--- log tail ---\n{open(logf).read()[-3000:]}")

In [ ]:
@stage("7 summary: metric lines from logs")
def _():
    for f in sorted(glob.glob(f"{OUT}/*.log")):
        lines = [l for l in open(f, errors="ignore") if re.search(r"hits|mrr|\bmr\b|mean rank", l, re.I)]
        log(f"== {os.path.basename(f)}: {len(lines)} metric lines; last 8:")
        log("".join(lines[-8:]))
    log("\nDONE. Everything is in /kaggle/working/report.txt (+ *.log). Send me report.txt.")